# OCPQ LoRA Model Serving on RWTH HPC

This notebook guides you through:
1. Uploading your NLxOCPQ code to RWTH HPC
2. Submitting a SLURM job to run the inference server
3. Tunneling the server back to your local machine
4. Configuring Streamlit UI to use the remote model

**Prerequisites:**
- SSH access to `rwth-hpc` with 2FA enabled
- Git LFS or manual upload of the adapter (lora_model_ocpq/)
- ~10-15 GB free space on HPC home directory

## Step 1: Upload Project to RWTH HPC

Run these commands on your **local machine** to upload the NLxOCPQ project to HPC.

In [ ]:
# Run locally: Upload NLxOCPQ folder and Logs to HPC
# Adjust paths if needed

! scp -r ~/Desktop/RWTH/Thesis/Bread/NLxOCPQ rwth-hpc:~/
! echo "Upload complete. Verify on HPC with: ssh rwth-hpc 'ls -la ~/NLxOCPQ'"

## Step 2: SSH into RWTH HPC and Check Job Partition

Open a terminal and run:

In [ ]:
# Run in HPC terminal:
# ssh rwth-hpc
# sinfo -p c23g   # Check GPU partition availability

## Step 3: Submit SLURM Job

From the HPC login node, submit the SLURM job script:

In [ ]:
# Run on HPC login node:
# cd ~
# sbatch NLxOCPQ/hpc_serve.slurm
# 
# Check job status:
# squeue -u $USER

## Step 4: Monitor Server Startup

Watch the server logs to confirm it started successfully:

In [ ]:
# Run on HPC login node:
# tail -f logs/JOBID_serve.log
#
# Expected output:
# Starting OCPQ LoRA inference server on node123 at ...
# INFO:     Uvicorn running on http://127.0.0.1:8000
#
# Note the node name (e.g., node123) for tunneling

## Step 5: Create SSH Tunnel to Your Local Machine

In a **new terminal on your local machine**, create a reverse port forward:

```bash
ssh -N -L 8000:node123:8000 rwth-hpc
```

Replace `node123` with the actual compute node name from Step 4.

This tunnel will remain open as long as the SSH session is active.

## Step 6: Test the Tunnel

In a **new terminal on your local machine**, test that the tunnel is working:

In [ ]:
# Test tunnel health endpoint
! curl -v http://localhost:8000/health

Expected response:
```json
{"status":"ok","model":"unsloth/llama-3-8b-Instruct-bnb-4bit"}
```

## Step 7: Test Inference via Tunnel

Send a test inference request through the tunnel:

In [ ]:
import requests
import json

# Test endpoint via tunnel
url = "http://localhost:8000/chat/completions"

payload = {
    "model": "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "messages": [
        {"role": "user", "content": "Hello, world!"}
    ],
    "max_tokens": 100,
    "temperature": 0.7
}

response = requests.post(url, json=payload, timeout=30)
print(f"Status: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

## Step 8: Configure Streamlit UI for Remote Model

On your **local machine**, start Streamlit with the custom endpoint:

In [ ]:
# Run locally:
# cd ~/Desktop/RWTH/Thesis/Bread
# export HF_BASE_URL=http://localhost:8000
# streamlit run NLxOCPQ/ui.py

In the Streamlit sidebar:
1. Select **"Custom endpoint"**
2. Set **Endpoint base URL** to: `http://localhost:8000`
3. Leave **Endpoint auth token** blank (or fill if your server requires auth)
4. Click "Generate & Execute Query" with your NL prompt

## Step 9: Cleanup

When you're done, clean up resources:

In [ ]:
# On HPC login node: Cancel the SLURM job
# scancel JOBID
#
# In local terminal running tunnel: Press Ctrl+C
#
# Stop Streamlit: Press Ctrl+C in Streamlit terminal

## Troubleshooting

**Q: Connection refused when testing tunnel?**
- Verify server started: `tail logs/JOBID_serve.log` on HPC
- Confirm tunnel is active in your local terminal
- Re-check the node name in `squeue -u $USER`

**Q: OutOfMemoryError in server logs?**
- Reduce `max_tokens` sent from Streamlit
- c23g GPUs (A100 80GB) should handle llama-3-8b with 4-bit loading
- If persistent, use a smaller model or enable more aggressive memory optimization

**Q: 404 on /chat/completions?**
- Ensure tunnel base URL is exactly `http://localhost:8000` (no trailing slash)
- Server supports both `/chat/completions` and `/v1/chat/completions`

For more details, see `HPC_RWTH_SETUP.md` in the NLxOCPQ directory.